# Phase 6D - Prototype Comparison

Notebook ini membandingkan hasil smoke test IndoBERTweet, Temporal LSTM, dan GraphSAGE. Semua angka adalah hasil split awal pada 100 akun berlabel dan belum menjadi hasil final.


In [1]:
from pathlib import Path
import json
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'PRD.md').exists())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
OUTPUT = ROOT / 'output' / 'phase6_prototypes'
print('Repository:', ROOT)


Repository: /home/andry-alfisi/Project/Project/Buzzer-Detection-


In [2]:
model_dirs = {'IndoBERTweet': 'indobertweet', 'Temporal LSTM': 'temporal_lstm', 'GraphSAGE': 'graphsage'}
rows = []
for model_name, directory in model_dirs.items():
    readiness_path = OUTPUT / directory / 'readiness.json'
    readiness = json.loads(readiness_path.read_text(encoding='utf-8'))
    rows.append({'model': model_name, **readiness['metrics'], 'labeled_accounts': readiness['data'].get('labeled_accounts', readiness['data'].get('labeled_nodes')), 'notes': ' | '.join(readiness['notes'])})
comparison = pd.DataFrame(rows)
comparison[['model', 'n', 'accuracy', 'precision_buzzer', 'recall_buzzer', 'f1_buzzer', 'roc_auc', 'pr_auc', 'labeled_accounts']].to_csv(OUTPUT / 'prototype_metrics_comparison.csv', index=False)
comparison[['model', 'n', 'accuracy', 'precision_buzzer', 'recall_buzzer', 'f1_buzzer', 'roc_auc', 'pr_auc', 'labeled_accounts']]


,model,n,accuracy,precision_buzzer,recall_buzzer,f1_buzzer,roc_auc,pr_auc,labeled_accounts
0,IndoBERTweet,30,0.800000,0.812500,0.8125,0.812500,0.875000,0.872381,100
1,Temporal LSTM,30,0.700000,0.705882,0.7500,0.727273,0.758929,0.715739,100
2,GraphSAGE,30,0.666667,0.666667,0.7500,0.705882,0.700893,0.766504,100


In [3]:
metric_columns = ['accuracy', 'precision_buzzer', 'recall_buzzer', 'f1_buzzer', 'roc_auc', 'pr_auc']
axis = comparison.set_index('model')[metric_columns].plot(kind='bar', figsize=(11, 5), ylim=(0, 1), rot=0)
axis.set_title('Phase 6 Prototype Metrics - Initial Smoke Test')
axis.set_ylabel('Score')
axis.set_xlabel('')
axis.grid(axis='y', alpha=0.25)
axis.legend(loc='lower right')
figure = axis.get_figure()
figure.tight_layout()
figure.savefig(OUTPUT / 'prototype_metrics_comparison.png', dpi=180)
plt.close(figure)
print('Saved:', OUTPUT / 'prototype_metrics_comparison.png')


Saved: /home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase6_prototypes/prototype_metrics_comparison.png


In [4]:
assert len(comparison) == 3
assert comparison['model'].is_unique
assert (comparison['n'] > 0).all()
assert (OUTPUT / 'prototype_metrics_comparison.csv').exists()
assert (OUTPUT / 'prototype_metrics_comparison.png').exists()
print('Prototype comparison validation: OK')


Prototype comparison validation: OK


## Cara membaca hasil

- IndoBERTweet menguji representasi teks; hasil tinggi pada data kecil belum membuktikan generalisasi lintas isu.
- LSTM menguji urutan waktu; tambah label multi-post sebelum tuning serius.
- GraphSAGE menguji struktur relasi; lanjutkan dengan cross-issue dan edge holdout.
- Jangan memilih model hanya dari satu split; gunakan group, temporal, cross-issue, dan ablation setelah data bertambah.
